# Multi-objective Evolution: Class Activity
**DCS 340 · Max Hoffman · Nov 10, 2026**

Running example (from my summer research in the Bavis lab): a newborn rat in low oxygen has two goals that fight. Breathing more brings in O₂, but breathing and keeping warm cost energy, which costs O₂. Pups answer by raising ventilation and then lowering metabolic rate. There is no single best response. There is a *set* of best trade-offs.

Parts:
- **A.** Be the selection operator: find the Pareto front by hand (5 min)
- **B.** Why not just add the objectives together? (5 min)
- **C.** Run NSGA-II, then break it on purpose (8 min)
- **D.** Discussion

Run each cell top to bottom. Nothing to install.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (5.5, 4)

## Part A: Be the selection operator

Ten made-up neonatal rat pups in 12% O₂. We want **high** ventilation (O₂ supply) and **low** energy cost (O₂ spent on breathing and heat).

Rat X **dominates** rat Y if X is at least as good on both objectives and strictly better on at least one.

**Task:** before running the next cell, write down which rats are *not dominated by anyone*. Then check yourself.

*(Numbers are illustrative, not data from my lab.)*

In [ ]:
rats = {
 'A': (1.48, 3.2), 'B': (1.85, 3.6), 'C': (2.00, 4.6), 'D': (1.70, 4.0),
 'E': (2.25, 5.9), 'F': (1.50, 3.0), 'G': (1.90, 4.8), 'H': (1.95, 5.0),
 'I': (1.40, 3.3), 'J': (2.30, 7.2),
}  # name: (ventilation, mL/min/g -> maximize ; energy cost, arbitrary units -> minimize)

def dominates(a, b):
    # a, b = (ventilation, cost). Higher ventilation better, lower cost better.
    better_or_equal = a[0] >= b[0] and a[1] <= b[1]
    strictly_better = a[0] > b[0] or a[1] < b[1]
    return better_or_equal and strictly_better

pareto = [n for n, v in rats.items()
          if not any(dominates(w, v) for m, w in rats.items() if m != n)]
print('Non-dominated rats:', pareto)

for n, (c, v) in rats.items():
    plt.scatter(v, c, c='tab:red' if n in pareto else 'lightgray', s=60, zorder=3)
    plt.annotate(n, (v, c), xytext=(5, 3), textcoords='offset points')
front = sorted([rats[n] for n in pareto], key=lambda p: p[1])
plt.plot([p[1] for p in front], [p[0] for p in front], 'r--', lw=1)
plt.xlabel('Energy cost (a.u.)  ← better'); plt.ylabel('Ventilation (mL/min/g)  better →')
plt.title('Part A: the Pareto front'); plt.show()

**Check-in:** which rat surprised you? Is pup **J** (breathes the most, costs the most) "bad"? Is pup **F**?

Neither is dominated. Picking *between* front members needs information the objectives don't contain (a physiologist's judgment about what the pup needs right now, a decision-maker's preference).

## Part B: Why not just add the objectives together?

The obvious shortcut is a **weighted sum**: minimize `w·f1 + (1−w)·f2`. Sweep `w` from 0 to 1 and collect the winners.

Below are two fronts with the same endpoints. One bulges toward the ideal corner (convex), one sags away from it (concave).

In [ ]:
t = np.linspace(0, 1, 200)
fronts = {'convex front': np.c_[t, 1 - np.sqrt(t)],
          'concave front': np.c_[t, 1 - t**2]}

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, F) in zip(axes, fronts.items()):
    winners = set()
    for w in np.linspace(0, 1, 101):
        winners.add(int(np.argmin(w * F[:, 0] + (1 - w) * F[:, 1])))
    W = F[sorted(winners)]
    ax.plot(F[:, 0], F[:, 1], color='lightgray', lw=3, label='true front')
    ax.scatter(W[:, 0], W[:, 1], c='tab:red', zorder=3, label=f'weighted-sum hits ({len(winners)} distinct)')
    ax.set_title(name); ax.set_xlabel('f1 (minimize)'); ax.set_ylabel('f2 (minimize)'); ax.legend()
plt.tight_layout(); plt.show()

**Question:** 101 different weights. How many distinct solutions did the weighted sum find on the concave front? Why?

*Hint:* a weighted sum is a straight line sliding toward the corner. Which points can a straight line touch first on a curve that sags?

## Part C: NSGA-II from scratch

NSGA-II (Deb et al., 2002) changes only **survivor selection** in a normal GA:
1. **Non-dominated sorting:** rank 1 = Pareto front of the population, rank 2 = front once rank 1 is removed, ...
2. **Crowding distance:** inside a rank, prefer individuals in empty regions (keeps the front spread out).
3. **Elitism:** parents + children compete together; keep the best N.

Test problem: **ZDT1** (Zitzler, Deb & Thiele, 2000), 30 variables in [0,1], two objectives, true front `f2 = 1 − √f1`.

In [ ]:
def zdt1(X):
    f1 = X[:, 0]
    g = 1 + 9 * X[:, 1:].mean(axis=1)
    f2 = g * (1 - np.sqrt(f1 / g))
    return np.c_[f1, f2]

def nondominated_sort(F):
    n = len(F)
    dominated_by = [[] for _ in range(n)]   # who i dominates
    counts = np.zeros(n, int)                # how many dominate i
    for i in range(n):
        le = np.all(F[i] <= F, axis=1) & np.any(F[i] < F, axis=1)   # i dominates j
        dominated_by[i] = list(np.where(le)[0])
        counts[i] = np.sum(np.all(F <= F[i], axis=1) & np.any(F < F[i], axis=1))
    fronts, current = [], list(np.where(counts == 0)[0])
    while current:
        fronts.append(current)
        nxt = []
        for i in current:
            for j in dominated_by[i]:
                counts[j] -= 1
                if counts[j] == 0:
                    nxt.append(j)
        current = nxt
    return fronts

def crowding_distance(F):
    n, m = F.shape
    d = np.zeros(n)
    for k in range(m):
        order = np.argsort(F[:, k])
        d[order[0]] = d[order[-1]] = np.inf          # always keep the extremes
        span = F[order[-1], k] - F[order[0], k] or 1.0
        d[order[1:-1]] += (F[order[2:], k] - F[order[:-2], k]) / span
    return d

In [ ]:
def vary(P, eta=15, pm=None):
    # simple variation: uniform crossover + gaussian mutation
    n, d = P.shape
    pm = pm or 1 / d
    idx = rng.integers(0, n, (n, 2))
    a, b = P[idx[:, 0]], P[idx[:, 1]]
    u = rng.random((n, d))
    C = np.where(u < 0.5, a, b) + rng.normal(0, 0.1, (n, d)) * (rng.random((n, d)) < pm)
    return np.clip(C, 0, 1)

def nsga2(problem, n_var=30, pop=100, gens=150, use_crowding=True, seed=0):
    global rng
    rng = np.random.default_rng(seed)
    P = rng.random((pop, n_var))
    for _ in range(gens):
        R = np.vstack([P, vary(P)])              # parents + children (elitism)
        F = problem(R)
        survivors = []
        for front in nondominated_sort(F):
            if len(survivors) + len(front) <= pop:
                survivors += front
            else:
                need = pop - len(survivors)
                if use_crowding:
                    cd = crowding_distance(F[front])
                    pick = np.argsort(-cd)[:need]
                else:
                    pick = rng.choice(len(front), need, replace=False)   # no diversity pressure
                survivors += list(np.array(front)[pick])
                break
        P = R[survivors]
    return problem(P)

F_on = nsga2(zdt1)
t = np.linspace(0, 1, 200)
plt.plot(t, 1 - np.sqrt(t), color='lightgray', lw=3, label='true front')
plt.scatter(F_on[:, 0], F_on[:, 1], s=12, c='tab:red', label='NSGA-II, gen 150')
plt.xlabel('f1'); plt.ylabel('f2'); plt.legend(); plt.title('ZDT1'); plt.show()

### Break it on purpose

Each pair takes one change, runs it, and reports what happened to the front. Edit the call in the next cell.

| Change | Edit |
|---|---|
| 1. Turn off crowding distance | `use_crowding=False` |
| 2. Tiny population | `pop=10` |
| 3. Fewer generations | `gens=20` |
| 4. Different seed (is it luck?) | `seed=3` |
| 5. Harder problem: concave front | swap in `zdt2` (defined below) |

Prediction first: write one sentence about what you expect *before* running.

In [ ]:
def zdt2(X):
    f1 = X[:, 0]
    g = 1 + 9 * X[:, 1:].mean(axis=1)
    f2 = g * (1 - (f1 / g) ** 2)
    return np.c_[f1, f2]

# ---- EDIT THIS LINE ----
F_test = nsga2(zdt1, use_crowding=False)
# ------------------------

plt.plot(t, 1 - np.sqrt(t), color='lightgray', lw=3, label='ZDT1 true front')
plt.plot(t, 1 - t**2, color='lightgray', lw=3, ls=':', label='ZDT2 true front')
plt.scatter(F_on[:, 0], F_on[:, 1], s=10, c='tab:red', alpha=.35, label='baseline')
plt.scatter(F_test[:, 0], F_test[:, 1], s=14, c='tab:blue', label='your change')
plt.xlabel('f1'); plt.ylabel('f2'); plt.legend(fontsize=8); plt.show()

# a crude spread score: how much of the f1 axis does the front cover?
print('f1 coverage  baseline: %.2f   yours: %.2f' % (np.ptp(F_on[:, 0]), np.ptp(F_test[:, 0])))

## Part D: Discussion

1. In Part A, the algorithm can't tell you whether pup **E** or pup **B** is "better." Who should make that call, and when: before the run (weights) or after (pick from the front)?
2. Evolution in real organisms has no weights either. A rat pup in hypoxia first breathes more, then cuts its metabolic rate. Is that one fixed weighting, or a move along the front over time?
3. Thursday's reading (Lehman & Stanley) argues for dropping objectives entirely. Multi-objective EAs already give up on a single "best." Is that a step toward their argument, or a different idea?

### References
- Eiben, A. E. & Smith, J. E. *Introduction to Evolutionary Computing*. Multiobjective EAs (Ch. 9 in 1st ed.; Ch. 12 in 2nd ed., 2015).
- Deb, K., Pratap, A., Agarwal, S. & Meyarivan, T. (2002). A fast and elitist multiobjective genetic algorithm: NSGA-II. *IEEE Trans. Evolutionary Computation* 6(2), 182–197.
- Zitzler, E., Deb, K. & Thiele, L. (2000). Comparison of multiobjective evolutionary algorithms: empirical results. *Evolutionary Computation* 8(2), 173–195.
- Shoval, O. et al. (2012). Evolutionary trade-offs, Pareto optimality, and the geometry of phenotype space. *Science* 336, 1157–1160.